# 02 — ObjectScale S3 Throttler
**Objective:** validate native S3 throttling — a noisy tenant is capped while a critical workload stays inside its SLA and recovers.

**Customer sees:** measured phases with PASS / FAIL / INCONCLUSIVE verdicts against approved thresholds, and a live TPS/latency chart.
**TC:** TC2.1–TC2.11

**Flow:** rules are verified and stored first (throttler still disabled) so they have fully propagated before enforcement is measured → Phase 1 baseline (critical only) → Phase 2 contention (+noisy, throttler disabled) → Phase 3 simulation (dry-run) → propagation wait → Phase 4 enforcement (HTTP 429) → Phase 5 recovery observation (noisy **keeps running**) → prefix-scoped rule → Spark write as the throttled tenant → restore.

> ObjectScale caches the rule set for up to `rules_config_cache_expiration_time` = 300 s (read-only). The notebook therefore waits `PROPAGATION_S` (default 300 s) after storing the rules before it measures enforcement. A verdict the evidence cannot decide is INCONCLUSIVE — never a pass.

## Step 1 — Session, tenant isolation, effective policy
Prove noisy/critical are distinct IAM principals, then **read** the current throttler state. The pre-demo backup must exist before anything is changed.

In [ ]:
import atexit, json, os, time
import ipywidgets as w
import pandas as pd
from IPython.display import HTML, display
from poc import config, results, viz
from poc import throttle_eval as te
from poc.loadgen import LoadGen, put_burst
from poc.obs_admin import MgmtApi, Throttler
import poc.spark_helpers as sh

spark = sh.build_session("obs-poc")
viz.table(pd.DataFrame(sh.versions(spark).items(), columns=["component", "version"]))

PHASE_S = int(os.environ.get("THROTTLE_PHASE_S", "60"))
PROPAGATION_S = int(os.environ.get("PROPAGATION_S", "300"))
SETTLE_S = int(os.environ.get("THROTTLE_SETTLE_S", "120"))
DUR = {"baseline": PHASE_S, "contention": PHASE_S, "simulation": PHASE_S,
       "enforced": 2 * PHASE_S, "recovery": PHASE_S,
       "burst": max(15, PHASE_S // 3)}
RID = results.run_id()

lg = LoadGen(max_keys=20000)
iso_rows, iso_ok = lg.verify_tenants()
viz.table(pd.DataFrame(iso_rows))
thr = Throttler(MgmtApi())
pol = te.effective_policy(thr)
viz.table(pd.DataFrame([{"mode": pol["mode"], "response_code": pol["response_code"],
                         "existing_rules": len(pol["rules"]),
                         "backup": str(config.BACKUP_FILE),
                         "backup_exists": config.BACKUP_FILE.exists()}]))
if not config.BACKUP_FILE.exists():
    raise RuntimeError(f"{config.BACKUP_FILE} is missing — run scripts/prep_obs.py "
                       "while the throttler is in its pre-demo state")
if pol["mode"] == "enabled":
    raise RuntimeError("the throttler is already enforcing — this demo must start "
                       "from mode disabled/dry-run (restore the pre-demo state first)")
print("run id:", RID, "| phase seconds:", DUR, "| PROPAGATION_S:", PROPAGATION_S)
viz.banner(iso_ok, "tenant isolation",
           "distinct principals; effective policy read (unchanged)")

**Presenter notes** (hidden)

Shared namespace bucket, separate principals — that is what principal-scoped rules discriminate on. LoadGen talks to S3 with SDK retries OFF so every 429 is seen as it arrives.

## Step 2 — Lab rules: schema verification, then store (TC2.2)
Two rules for the noisy tenant: all PUTs capped at 50 TPS, and a tighter 10 TPS cap on one key prefix (`<run_id>/raw/*`). `verify_rule` checks both against the schema and the **live** action list. Only then are they stored — with the throttler mode left untouched (disabled), so nothing is enforced yet. The time of the push starts the propagation clock.

In [ ]:
NOISY = f"urn:ecs:iam::{config.OBS_NAMESPACE}:user/tenant-noisy"
LIMIT, PREFIX_LIMIT = 50, 10
COVERED, CONTROL = f"{RID}/raw", f"{RID}/curated"
NOISY_RULE = {"id": "demo-noisy-put", "rateUnit": "TPS", "isDisabled": False,
              "filters": {"principals": [NOISY], "actions": ["s3:PutObject"]},
              "limit": {"value": LIMIT}, "scoping": {"per": ["principal"]}}
PREFIX_RULE = {"id": "demo-raw-prefix", "rateUnit": "TPS", "isDisabled": False,
               "filters": {"principals": [NOISY], "actions": ["s3:PutObject"],
                           "paths": {"matcher": "wildcard", "values": [
                               f"{config.OBS_NAMESPACE}/{config.BUCKET_THROTTLE}/{COVERED}/*"]}},
               "limit": {"value": PREFIX_LIMIT}, "scoping": {"per": ["principal"]}}
checks = {r["id"]: te.verify_rule(thr, r) for r in (NOISY_RULE, PREFIX_RULE)}
viz.table(pd.DataFrame([{"rule": rid, "schema_ok": c["ok"],
                         "problems": "; ".join(c["problems"]) or "—"}
                        for rid, c in checks.items()]))
display(HTML("<pre>" + json.dumps([NOISY_RULE, PREFIX_RULE], indent=2) + "</pre>"))
rules_ok = all(c["ok"] for c in checks.values())
res = results.write("TC2.2", "rules verified against the live schema", rules_ok,
                    {"rules": {rid: c["problems"] for rid, c in checks.items()}})
viz.banner(res["status"], "TC2.2", "rule schema + actions verified vs live API")
if not rules_ok:
    raise RuntimeError("a rule failed verification — nothing was pushed")

atexit.register(thr.restore_logged)          # restore even if a later cell dies
keep = [r for r in pol["rules"] if not str(r.get("id", "")).startswith("demo-")]
thr.put_config(keep + [NOISY_RULE, PREFIX_RULE])
PUSHED_AT = time.time()
print(f"rules stored at {time.strftime('%H:%M:%S', time.localtime(PUSHED_AT))} "
      f"(mode still {thr.get_mode()}); enforcement is measured no earlier than "
      f"{PROPAGATION_S}s after this push")

**Presenter notes** (hidden)

principals accept user:<name>, iam::user:<id>, or the full IAM URN. paths are <namespace>/<bucket>/<key-glob>; the wildcard spans '/'. The prefix rule is scoped per principal — scoping per 'path' would give every object key its own counter. Existing non-demo rules are kept in the pushed set.

## Step 3 — Phase 1: baseline (TC2.1)
Critical GETs alone — reference TPS and P50/P95/P99. Rules are stored but the throttler is disabled.

In [ ]:
out = w.Output(); display(out)
lg.start(noisy=False, critical=True)
lg.start_phase("baseline")
chart = viz.LiveChart(lg); chart.mark("baseline")
chart.refresh(out, seconds=DUR["baseline"])
bp = lg.phase_stats()["baseline"].get("tenant-critical GET", {})
res = results.write("TC2.1", "baseline critical GETs",
                    bp.get("ok", 0) >= 50 and bp.get("errors", 0) == 0,
                    {k: round(bp.get(k, 0), 2) for k in ("n", "tps", "p50", "p95", "p99", "errors")})
viz.banner(res["status"], "TC2.1 baseline",
           f"{bp.get('tps', 0):.1f} TPS, p95 {bp.get('p95', 0):.0f} ms, "
           f"p99 {bp.get('p99', 0):.0f} ms")

**Presenter notes** (hidden)

Critical workload alone — the reference everything else is compared with.

## Step 4 — Phase 2: contention, throttler disabled (TC2.3)
Noisy PUTs are added on top of the critical GETs. The throttler is still disabled, so this is the unprotected impact the rule must fix.

In [ ]:
lg.start_phase("contention"); chart.mark("contention"); lg.add_noisy(threads=6)
chart.refresh(out, seconds=DUR["contention"])
ps_all = lg.phase_stats()
c = ps_all["contention"]
bl = ps_all["baseline"].get("tenant-critical GET", {})
ct, cn = c.get("tenant-critical GET", {}), c.get("tenant-noisy PUT", {})
if min(ct.get("n", 0), cn.get("n", 0)) < 50:
    st3, why3 = "FAIL", "too few requests in the contention phase"
elif cn.get("throttled", 0):
    st3, why3 = "FAIL", f"{cn['throttled']} throttle responses while the throttler is disabled"
else:
    st3, why3 = "PASS", (f"critical p95 {bl.get('p95', 0):.0f} -> {ct.get('p95', 0):.0f} ms "
                         f"with {cn.get('tps', 0):.0f} noisy PUT/s")
res = results.write("TC2.3", "contention (throttler disabled)", status=st3, detail=why3,
    metrics={"baseline_p95": round(bl.get("p95", 0), 1),
             "contention_p95": round(ct.get("p95", 0), 1),
             "contention_p99": round(ct.get("p99", 0), 1),
             "critical_tps": round(ct.get("tps", 0), 1),
             "noisy_ok_tps": round(cn.get("ok", 0) / max(1e-9, c["_window"]["seconds"]), 1),
             "noisy_throttled": cn.get("throttled", 0), "mode": thr.get_mode(),
             "client_cpu_pct": {"baseline": ps_all["baseline"]["_window"]["client_cpu_pct"],
                                "contention": c["_window"]["client_cpu_pct"]}})
viz.banner(st3, "TC2.3 contention", why3)

**Presenter notes** (hidden)

Unrestricted noisy PUTs degrade critical latency — the problem statement. Caveat for this lab: both tenants are driven from one 2-vCPU client host, and a CPU-only disturbance on that host (no S3 traffic) degrades the critical GETs just as much — so the size of this effect cannot be attributed to ObjectScale alone (reports/environment.md, 'Client-side contention'). client_cpu_pct is recorded per phase.

## Step 5 — Phase 3: simulation mode, dry-run (TC2.10)
Mode **dry-run** (ObjectScale simulation mode): the server only counts rule matches. Both workloads continue — the client must see **zero** throttle responses.

In [ ]:
thr.set_mode("dryrun")
lg.start_phase("simulation"); chart.mark("simulation (dry-run)")
chart.refresh(out, seconds=DUR["simulation"])
sp = lg.phase_stats()["simulation"]
sn = sp.get("tenant-noisy PUT", {})
srv_sim = te.server_counters(thr)
if sn.get("n", 0) < 50:
    st10, why10 = "INCONCLUSIVE", "too few noisy requests in the simulation phase"
elif sn.get("throttled", 0):
    st10, why10 = "FAIL", f"{sn['throttled']} client throttle responses in dry-run"
else:
    st10, why10 = "PASS", (f"0 throttle responses in {sn['n']} noisy requests "
                           f"({sn.get('errors', 0)} other errors)")
res = results.write("TC2.10", "simulation mode (dry-run)", status=st10, detail=why10,
    metrics={"client_throttle_responses": sn.get("throttled", 0),
             "client_other_errors": sn.get("errors", 0),
             "noisy_ok_tps": round(sn.get("ok", 0) / max(1e-9, sp["_window"]["seconds"]), 1),
             "mode": thr.get_mode(),
             "server_metrics_available": srv_sim["available"]})
viz.banner(st10, "TC2.10 simulation", why10)

**Presenter notes** (hidden)

Dry-run validates rule matching safely. Simulated-match counters are a server-side concept: they are probed and reported as unavailable when the build does not expose them — never invented.

## Step 6 — Wait for rule propagation
ObjectScale caches the rule set for up to 300 s. Both workloads keep running (still dry-run) until `PROPAGATION_S` has passed since the rules were stored. This time is tagged as its own phase, `propagation`, so it does not pollute the measured phases.

In [ ]:
lg.start_phase("propagation"); chart.mark("propagation wait")
countdown = w.HTML(); display(countdown)
while (left := PROPAGATION_S - (time.time() - PUSHED_AT)) > 0:
    countdown.value = (f"<b>rule propagation: {left:.0f}s left</b> of {PROPAGATION_S}s "
                       f"since the rules were stored (load keeps running, dry-run)")
    chart.refresh(out, seconds=min(10, left))
RULE_AGE_S = round(time.time() - PUSHED_AT, 1)
countdown.value = f"<b>rules are {RULE_AGE_S:.0f}s old</b> — propagation wait complete"
print(f"rules stored {RULE_AGE_S:.0f}s ago (>= PROPAGATION_S={PROPAGATION_S}s) — "
      f"enforcement can now be measured")

**Presenter notes** (hidden)

The countdown is live in JupyterLab; headless runs print the final line. With the default 60 s phases roughly 120 s are spent here.

## Step 7 — Phase 4: enforcement, HTTP 429 (TC2.4)
Mode **enabled**, response code 429. The switch is read back, the notebook waits until the new mode is visibly in effect (tagged `switching`), then measures the `enforced` phase with both workloads running. The status is computed: PASS needs an enforcement signature **and** an accepted noisy rate within the limit.

In [ ]:
thr.set_response_code(429); thr.set_mode("enabled")
time.sleep(2)                                   # let the config write settle
eff = {"mode": thr.get_mode(), "response_code": thr.get_response_code(),
       "rules": [r.get("id") for r in thr.get_rules()]}
assert eff["mode"] == "enabled" and eff["response_code"] == 429, eff
lg.start_phase("switching"); chart.mark("mode -> enabled")
t_eff = te.wait_effective(lg, LIMIT, timeout_s=SETTLE_S)
print("mode switch visible after", t_eff, "s" if t_eff is not None
      else f"— NOT visible within {SETTLE_S}s; measuring anyway")

lg.start_phase("enforced"); chart.mark("enforced (429)")
half = DUR["enforced"] // 2
chart.refresh(out, seconds=half)
sustained_mid = lg.sustained(10, LIMIT * 0.25)
chart.refresh(out, seconds=DUR["enforced"] - half)
sustained_end = lg.sustained(10, LIMIT * 0.25)

ps_all = lg.phase_stats()
ep, cp = ps_all["enforced"], ps_all["contention"]
en, ec = ep.get("tenant-noisy PUT", {}), ep.get("tenant-critical GET", {})
st4, why4, enforced, enf_mode = te.tc24_status(
    ps_all, LIMIT, sustained=sustained_mid and sustained_end)
sig = te.signals(ps_all, LIMIT)
res = results.write("TC2.4", "enforcement (HTTP 429)", status=st4, detail=why4,
    metrics={"noisy": {k: round(en.get(k, 0), 2) for k in
                       ("n", "ok", "throttled", "r429", "r503_slowdown", "r503_other",
                        "errors", "retries_ok", "retries_fail", "boundary",
                        "ok_p50", "ok_p95", "ok_p99", "fail_p50", "fail_p95")},
             "noisy_ok_tps_before": round(sig["ok_tps_before"], 1),
             "noisy_ok_tps_enforced": round(sig["ok_tps"], 1),
             "limit_tps": LIMIT, "codes": en.get("codes", {}),
             "critical_tps": round(ec.get("tps", 0), 1),
             "critical_p95": round(ec.get("p95", 0), 1),
             "critical_p99": round(ec.get("p99", 0), 1),
             "critical_throttled": ec.get("throttled", 0),
             "critical_errors": ec.get("errors", 0),
             "sustained_mid": sustained_mid, "sustained_end": sustained_end,
             "rule_age_at_enable_s": RULE_AGE_S, "seconds_to_effect": t_eff,
             "effective_config": eff,
             "enforcement_active": enforced, "enforcement_mode": enf_mode})
viz.banner(st4, "TC2.4 enforcement", why4)

**Presenter notes** (hidden)

With client retries off ObjectScale shows both mechanisms: it queues requests to hold the accepted rate at the limit (latency tail grows) and rejects the overflow with 429 TooManyRequests. codes{} lists the S3 error codes exactly as received.

## Step 8 — Phase 5: continued recovery observation (TC2.5)
**No configuration change here** — enforcement from Step 7 stays active and this phase simply keeps measuring. Noisy PUTs continue at the same offered rate, the rule keeps capping them, and we observe whether critical latency holds near baseline. Throttle responses on the noisy side are expected to stay elevated.

In [ ]:
lg.start_phase("recovery"); chart.mark("recovery (enforcement continues)")
chart.refresh(out, seconds=DUR["recovery"])
ps_all = lg.phase_stats()
rp = ps_all["recovery"]
rc, rn = rp.get("tenant-critical GET", {}), rp.get("tenant-noisy PUT", {})
rec_span = max(1e-9, rp["_window"]["seconds"])
verdicts = te.evaluate(ps_all, rule=NOISY_RULE)
REC_CHECKS = {"recovery_p95_vs_baseline", "noisy_sustained_in_recovery",
              "critical_not_throttled"}
st5 = te.combine(verdicts["verdicts"], REC_CHECKS)
why5 = "; ".join(f"{v['name']}={v['verdict']}" for v in verdicts["verdicts"]
                 if v["name"] in REC_CHECKS)
res = results.write("TC2.5", "recovery observation under sustained throttling",
    status=st5, detail=why5,
    metrics={"baseline_p95": round(ps_all["baseline"].get("tenant-critical GET", {}).get("p95", 0), 1),
             "contention_p95": round(ps_all["contention"].get("tenant-critical GET", {}).get("p95", 0), 1),
             "critical_p95": round(rc.get("p95", 0), 1),
             "critical_p99": round(rc.get("p99", 0), 1),
             "critical_tps": round(rc.get("tps", 0), 1),
             "critical_errors": rc.get("errors", 0),
             "critical_throttled": rc.get("throttled", 0),
             "noisy_offered_tps": round(rn.get("n", 0) / rec_span, 1),
             "noisy_ok_tps": round(rn.get("ok", 0) / rec_span, 1),
             "noisy_throttled": rn.get("throttled", 0),
             "client_cpu_pct": {ph: ps_all[ph]["_window"]["client_cpu_pct"]
                                for ph in ("baseline", "contention", "enforced", "recovery")},
             "recovery_p95_dev_factor": verdicts["thresholds"]["recovery_p95_dev_factor"],
             "checks": {v["name"]: v["detail"] for v in verdicts["verdicts"]
                        if v["name"] in REC_CHECKS}})
viz.banner(st5, "TC2.5 recovery",
           f"critical p95 {rc.get('p95', 0):.0f} ms (baseline "
           f"{res['metrics']['baseline_p95']:.0f}, contention "
           f"{res['metrics']['contention_p95']:.0f}); noisy still offering "
           f"{rn.get('n', 0) / rec_span:.0f} req/s")

**Presenter notes** (hidden)

The mechanism is the already-running enforcement, observed longer — not a second action. The rule still admits ~50 noisy PUT/s by design, so the critical workload shares the system with that admitted load; the recovery threshold is the customer's to set (te.DEFAULT_THRESHOLDS).

## Step 9 — Prefix-scoped rule (TC2.6)
The second rule caps the noisy tenant at 10 TPS under `<run_id>/raw/*` only. Real traffic decides: the main noisy load is paused, then the same tenant sends one PUT burst to the covered prefix and one to a control prefix (`<run_id>/curated/`). PASS only if the covered prefix is held at its limit while the control prefix runs clearly above it (it stays under the tenant-wide 50 TPS rule).

In [ ]:
lg.pause_noisy(); time.sleep(3)
lg.start_phase("prefix-test"); chart.mark("prefix bursts")
cov = put_burst("noisy", config.BUCKET_THROTTLE, COVERED, seconds=DUR["burst"], threads=4)
ctl = put_burst("noisy", config.BUCKET_THROTTLE, CONTROL, seconds=DUR["burst"], threads=4)
pfx = te.evaluate_prefix(cov, ctl, PREFIX_LIMIT)
stored = [r.get("id") for r in thr.get_rules()]
if PREFIX_RULE["id"] not in stored:
    pfx = {"status": "FAIL", "reason": f"prefix rule not stored on the server: {stored}"}
cols = ("n", "ok", "ok_tps", "throttled", "r429", "errors", "ok_p95", "seconds")
viz.table(pd.DataFrame([{"prefix": "covered  " + COVERED, **{k: round(cov[k], 1) for k in cols}},
                        {"prefix": "control  " + CONTROL, **{k: round(ctl[k], 1) for k in cols}}]))
res = results.write("TC2.6", "prefix-scoped rule", status=pfx["status"], detail=pfx["reason"],
    metrics={"prefix_limit_tps": PREFIX_LIMIT, "tenant_limit_tps": LIMIT,
             "rule_path": PREFIX_RULE["filters"]["paths"]["values"][0],
             "covered": {k: round(cov[k], 1) for k in cols},
             "control": {k: round(ctl[k], 1) for k in cols},
             "covered_codes": cov["codes"], "control_codes": ctl["codes"]})
viz.banner(pfx["status"], "TC2.6 prefix rule", pfx["reason"])
lg.resume_noisy()

**Presenter notes** (hidden)

Both bursts use the tenant-noisy identity with retries off. The control prefix is still subject to the tenant-wide rule, so it is expected near 50 TPS, the covered prefix near 10.

## Step 10 — Spark write as the throttled tenant (TC2.7)
The same Spark job runs twice while the noisy load keeps the tenant at its limit: first as **poc-admin** (not targeted by the rule), then — in a session rebuilt with the **tenant-noisy** S3A identity — under the enforced rule. PASS: the throttled job completes and throttling was demonstrably active while it ran.

In [ ]:
lg.start_phase("spark-write"); chart.mark("spark writes")

def spark_write(session, label):
    target = f"{RID}/spark-write-{label}"
    t0 = time.time()
    session.range(200_000).repartition(20).write.mode("overwrite").json(
        f"s3a://{config.BUCKET_THROTTLE}/{target}")
    return round(time.time() - t0, 1), target

t_admin, _ = spark_write(spark, "admin")
spark.stop()
spark = sh.build_session("obs-poc-noisy", s3a_user="noisy")
ident = sh.identities(spark)
t_start = time.time()
try:
    t_noisy, target = spark_write(spark, "noisy"); job_error = None
except Exception as e:
    t_noisy, target, job_error = None, f"{RID}/spark-write-noisy", str(e).splitlines()[0][:300]
during = lg.snapshot(since=t_start).get("tenant-noisy", {}).get("PUT", {})
s3a_stats = sh.s3a_committer_stats(config.BUCKET_THROTTLE, target) if not job_error else {}
s3a_throttled = sum(v for k, v in s3a_stats.items()
                    if "throttl" in k and isinstance(v, (int, float)))
mode_now = thr.get_mode()
if job_error:
    st7, why7 = "FAIL", f"Spark write as tenant-noisy failed: {job_error}"
elif ident["s3a"] != "noisy":
    st7, why7 = "FAIL", f"session S3A identity is {ident['s3a']}, not tenant-noisy"
elif mode_now != "enabled" or not (during.get("throttled", 0) or s3a_throttled):
    st7, why7 = "INCONCLUSIVE", (f"job completed in {t_noisy}s, but no throttling was "
                                 f"observed for tenant-noisy while it ran (mode={mode_now})")
else:
    st7, why7 = "PASS", (f"completed under enforcement in {t_noisy}s vs {t_admin}s as "
                         f"poc-admin ({t_noisy / max(t_admin, 0.1):.1f}x); "
                         f"{during.get('throttled', 0)} tenant-noisy 429s during the job")
res = results.write("TC2.7", "spark write as throttled tenant", status=st7, detail=why7,
    metrics={"seconds_admin_unthrottled": t_admin, "seconds_noisy_throttled": t_noisy,
             "slowdown": round(t_noisy / max(t_admin, 0.1), 2) if t_noisy else None,
             "s3a_identity": ident["s3a"], "s3a_key": ident["s3a_key"],
             "mode": mode_now,
             "tenant_noisy_429_during_job": during.get("throttled", 0),
             "tenant_noisy_ok_tps_during_job": round(
                 during.get("ok", 0) / max(1e-9, time.time() - t_start), 1),
             "s3a_committer_stats": s3a_stats})
viz.table(pd.DataFrame([{"identity": "poc-admin (not in the rule)", "seconds": t_admin},
                        {"identity": "tenant-noisy (rule enforced)", "seconds": t_noisy}]))
viz.banner(st7, "TC2.7 Spark under throttling", why7)

**Presenter notes** (hidden)

The S3A identity is set per session (fs.s3a.access.key), verified by build_session. S3A retries throttled requests (fs.s3a.retry.throttle.*), so the job slows down instead of failing. The 429 count is the tenant's LoadGen view during the job; S3A's own counters are shown when the committer reports them.

## Step 11 — Stop the load: worker lifecycle (TC2.11)
Stop and restart the noisy group once to prove it restarts safely, then stop every worker.

In [ ]:
leftover = lg.stop_noisy()
restarted = lg.add_noisy(1) == 1
time.sleep(3)
restarted = restarted and lg.running("tenant-noisy")
leftover += lg.stop()
res = results.write("TC2.11", "worker lifecycle", restarted and leftover == 0,
                    {"noisy_restartable": restarted, "lingering_workers": leftover})
viz.banner(res["status"], "TC2.11 worker lifecycle",
           "restart ok, no lingering workers" if res["passed"]
           else f"restartable={restarted}, lingering={leftover}")

**Presenter notes** (hidden)

stop() joins every worker; a worker that outlives its join timeout is kept in the count instead of being forgotten.

## Step 12 — Verdicts + server-side correlation (TC2.9)
Score all phases against the approved thresholds and correlate the client-side throttle responses with ObjectScale's own counters where the build exposes them. The status **is** the overall verdict — INCONCLUSIVE is not a pass.

In [ ]:
ps_all = lg.phase_stats()
verdicts = te.evaluate(ps_all, rule=NOISY_RULE)
srv = te.server_counters(thr)
rows = [{"check": v["name"], "verdict": v["verdict"], "detail": v["detail"]}
        for v in verdicts["verdicts"]]
display(pd.DataFrame(rows).style.apply(viz.status_style, column="verdict", axis=1))
corr = {"client_throttled_enforced_phase": ps_all.get("enforced", {}).get(
            "tenant-noisy PUT", {}).get("throttled", 0),
        "server_metrics_available": srv["available"], "probed": srv["probed"]}
if srv["available"]:
    corr["server_data"] = srv["data"]
else:
    print("server-side throttle counters are not exposed on this build — "
          "client-side observations only")
res = results.write("TC2.9", "verdicts + correlation", status=verdicts["overall"],
                    metrics={"overall": verdicts["overall"],
                             "enforcement_mode": verdicts["enforcement_mode"],
                             "thresholds": verdicts["thresholds"],
                             "verdicts": rows, **corr})
viz.banner(verdicts["overall"], "TC2.9 overall verdict",
           f"enforcement mode: {verdicts['enforcement_mode']}")

**Presenter notes** (hidden)

A 50 TPS limit does not yield exactly 50 ok/s — the thresholds carry a stated tolerance. They are customer-approved values in te.DEFAULT_THRESHOLDS, not tuned to the result.

## Step 13 — Phase summary
Every phase side by side: request counts, TPS, latencies, throttle responses, errors.

In [ ]:
summary = [{"phase": ph, "client_cpu_pct": gs["_window"]["client_cpu_pct"],
            "workload": k, "n": v["n"], "tps": round(v["tps"], 1),
            "ok_p50": round(v["ok_p50"]), "ok_p95": round(v["ok_p95"]),
            "ok_p99": round(v["ok_p99"]), "throttled": v["throttled"],
            "r429": v["r429"], "errors": v["errors"], "boundary": v["boundary"]}
           for ph, gs in ps_all.items() for k, v in gs.items() if k != "_window"]
viz.table(pd.DataFrame(summary))
chart.refresh(out, seconds=2)

**Presenter notes** (hidden)

client_cpu_pct = CPU utilisation of the load-generator host in that phase. Shaded bands mark the phases; the dotted line is the throttled (429) rate. `boundary` counts requests that were still in flight when their phase ended.

## Step 14 — Restore (TC2.8)
Restore the pre-demo throttler backup and verify the live state against the backup file — mode, response code and rule IDs.

In [ ]:
backup = json.loads(config.BACKUP_FILE.read_text())
want = {"mode": backup["mode"], "response_code": int(backup["response_code"]),
        "rules": sorted(r.get("id") for r in backup["config"].get("throttle", {}).get("rules", []))}
thr.restore()
live = {"mode": thr.get_mode(), "response_code": thr.get_response_code(),
        "rules": sorted(r.get("id") for r in thr.get_rules())}
res = results.write("TC2.8", "restore pre-demo throttler state", live == want,
                    {"live": live, "backup": want})
viz.banner(res["status"], "TC2.8 restore",
           f"mode={live['mode']}, response_code={live['response_code']}, "
           f"rules={live['rules'] or 'none'}")
spark.stop()

**Presenter notes** (hidden)

Backup: reports/throttle_backup.json — written once by scripts/prep_obs.py from a clean pre-demo state. The restore is also registered at kernel exit.